In [2]:
import pandas as pd 
import numpy as np

In [3]:
def load_housing_data():
    return pd.read_csv("../data/housing/housing.csv")
housing=load_housing_data()

In [4]:
housing["income_cat"] = np.ceil(housing["median_income"] / 1.5)

housing["income_cat"] = housing["income_cat"].where(
    housing["income_cat"] < 5,
    5.0
)

from sklearn.model_selection import StratifiedShuffleSplit

split = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

for train_index, test_index in split.split(
    housing,
    housing["income_cat"]
):
    strat_train_set = housing.loc[train_index]
    strat_test_set = housing.loc[test_index]

for dataset in (strat_train_set, strat_test_set):
    dataset.drop("income_cat", axis=1, inplace=True)

In [10]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder , StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [6]:
exp_train = strat_train_set.drop("median_house_value", axis=1)
exp_train_labels = strat_train_set["median_house_value"].copy()

In [7]:
if "income_cat" in exp_train.columns:
    exp_train = exp_train.drop("income_cat", axis=1)

In [8]:
num_attribs = exp_train.drop("ocean_proximity", axis=1).columns.tolist()

cat_attribs = ["ocean_proximity"]

In [11]:
num_pipeline=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])

In [15]:
full_pipeline=ColumnTransformer([
    ("num",num_pipeline,num_attribs),
    ("cat",OneHotEncoder(),cat_attribs)
])

In [16]:
exp_train_prepare=full_pipeline.fit_transform(exp_train)

In [17]:
exp_train_prepare.shape

(16512, 13)